In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, numpy as np, pandas as pd, torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

bert_pipe = pipeline('fill-mask', model='bert-base-uncased')
gpt2_tok = AutoTokenizer.from_pretrained('gpt2')
gpt2_model = AutoModelForCausalLM.from_pretrained('gpt2')
gpt2_model.eval()

df = pd.read_csv('test.csv', header=None, names=['label','title','content']).dropna(subset=['content'])

In [ ]:
def get_longest_sent(text):
    parts = [s.strip() for s in re.split(r'[.!?]', str(text)) if s.strip()]
    return max(parts, key=lambda s: len(s.split())) if parts else str(text)

def clean(text, keep_punct=False):
    s = get_longest_sent(text).lower()
    s = re.sub(r'\d+', '', s)
    if not keep_punct:
        s = re.sub(r'[^\w\s]', '', s)
    return [t for t in s.split() if t.strip()]

def predict_bert(tokens, with_punct=False):
    if len(tokens) < 2: return '', ''
    true_w = re.sub(r'[^\w]', '', tokens[-1]).lower()
    prefix = ' '.join(tokens[:-1])
    if with_punct:
        prompt = prefix + ' [MASK].'
    else:
        prompt = prefix + ' [MASK]'
    try:
        pred = bert_pipe(prompt)[0]['token_str'].strip().lower()
    except:
        pred = ''
    return pred, true_w

def predict_gpt2(tokens):
    if len(tokens) < 2: return '', ''
    true_w = re.sub(r'[^\w]', '', tokens[-1]).lower()
    inputs = gpt2_tok(' '.join(tokens[:-1]), return_tensors='pt')
    with torch.no_grad():
        logits = gpt2_model(**inputs).logits[:, -1, :]
    pred = gpt2_tok.decode(torch.argmax(logits).item()).strip().lstrip().lower()
    return pred, true_w

In [ ]:
docs_np = [d for d in (clean(t, keep_punct=False) for t in df['content']) if len(d)>=2]
docs_wp = [d for d in (clean(t, keep_punct=True)  for t in df['content']) if len(d)>=2]

b_preds_np, g_preds_np, truths_np = [], [], []
for toks in docs_np:
    bp, tw = predict_bert(toks, with_punct=False)
    gp, _  = predict_gpt2(toks)
    b_preds_np.append(bp); g_preds_np.append(gp); truths_np.append(tw)

b_acc_np = sum(b==t for b,t in zip(b_preds_np, truths_np)) / len(truths_np)
g_acc_np = sum(g==t for g,t in zip(g_preds_np, truths_np)) / len(truths_np)
print(f'no punct  BERT={b_acc_np:.4f}  GPT2={g_acc_np:.4f}')

In [ ]:
b_preds_wp, g_preds_wp, truths_wp = [], [], []
for toks in docs_wp:
    bp, tw = predict_bert(toks, with_punct=True)
    gp, _  = predict_gpt2(toks)
    b_preds_wp.append(bp); g_preds_wp.append(gp); truths_wp.append(tw)

b_acc_wp = sum(b==t for b,t in zip(b_preds_wp, truths_wp)) / len(truths_wp)
g_acc_wp = sum(g==t for g,t in zip(g_preds_wp, truths_wp)) / len(truths_wp)
print(f'with punct  BERT={b_acc_wp:.4f}  GPT2={g_acc_wp:.4f}')

bert accuracy is really low without punctuation. this makes sense because bert is a masked model and needs words on both sides of [MASK] to predict well. when the mask is at the end theres nothing to the right so it just keeps predicting a period.

adding punctuation back helped a lot (went from 0.46% to 26.91%). putting a period after [MASK] gives bert the right-side context it was trained with which makes a big difference.

gpt2 does way better without punctuation since its designed to predict the next word from the left side only which is exactly this task. adding punctuation gives it a small boost too (24.20%) since it saw punctuation during training.